In [1]:
    # Pour faire des requêtes
import requests, time, json

    # Pour paralléliser des tâches
from concurrent.futures import ThreadPoolExecutor, as_completed
import multiprocessing
from tqdm import tqdm

    # Pour écrire des logs
import logging
logging.basicConfig(level=logging.INFO)


    # Librairie pour faire des tests sur les dictionnaires 
from collections import defaultdict

In [19]:
# import requests

CAMERAS_URL = (
    "https://s3-us-west-2.amazonaws.com/alertwildfire-data-public/all_cameras.json"
)
# METADA PAS TOUTES NECESSIARES POUR EFFECTUER LA REQUETE GET
HEADERS = {
    "Connection": "keep-alive",
    "Sec-Fetch-Site": "same-site",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Dest": "empty",
    "Referer": "https://www.alertwildfire.org/",
    "Host": "s3-us-west-2.amazonaws.com",
}

# Fréquence de scrapping
DURATION = "1mn"  # options: 15mn, 1h, 3h, 6h, 12h

# Temps d'attente pour recevoir une réponse d'une requête
MAX_TIME = 100

# A ce stade on a toutes les informations sur le .json
response = requests.get(CAMERAS_URL, headers=HEADERS)
cameras_data = response.json()
# print(cameras_data)
cameras_data["features"][100]["properties"]["name"]



'Cohasset Hill 2'

In [ ]:
# A L'EXECUTION DE CE CODE IL FAUT SCROLLER MANUELLEMENT SUR L'ENSEMBLE DES CAMERAS DE LA PAGE WEB QUI S'OUVRE POUR AVOIR TOUS LES UUIDS

from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import re
# import pandas as pd
import time

# --- Configuration Firefox headless ---
options = Options()
options.headless = True
driver = webdriver.Firefox(options=options)

try:
    # --- Charger la page Grid ---
    url = "https://www.alertwildfire.org/?viewMode=Grid"
    driver.get(url)

    # --- Attente explicite que les blocs caméra soient visibles ---
    wait = WebDriverWait(driver, 15)  # max 15s
    camera_blocks = wait.until(
        EC.visibility_of_all_elements_located(
            (By.CSS_SELECTOR, "div.firecam-viewer-wrapper div.row.row-cols-3.no-gutters div.col")
        )
    )

    # --- Table pour stocker résultats ---
    camera_table = []

    for block in camera_blocks:
        try:
            # Nom de la caméra
            header_elem = block.find_element(By.CSS_SELECTOR, "h3.block-title.text-truncate")
            camera_name = header_elem.text.strip()

            # Image contenant l'UUID
            img_elem = block.find_element(By.CSS_SELECTOR, "img.img-fluid")
            img_src = img_elem.get_attribute("src")

            # Extraire l'UUID depuis le src
            match = re.search(r'/([a-f0-9\-]{36})/latest_thumb\.jpg', img_src)
            camera_uuid = match.group(1) if match else None

            camera_table.append({
                "name": camera_name,
                "uuid": camera_uuid
            })
        except Exception:
            continue  # Ignorer les blocs sans image ou header

    # Convertir en DataFrame
    print(camera_table)
    # df_cameras = pd.DataFrame(camera_table)
    # print(df_cameras)

finally:
    driver.quit()


[{'name': 'NORTH MOKELUMNE 1', 'uuid': '5764b4bd-1b23-11f0-8845-02420a0001ad'}, {'name': 'HAWKINS PEAK 1', 'uuid': '575a4303-1b23-11f0-8845-02420a0001ad'}, {'name': 'LEEK SPRINGS 1', 'uuid': '575ee718-1b23-11f0-8845-02420a0001ad'}, {'name': 'ARMSTRONG LOOKOUT 1', 'uuid': '574eb94d-1b23-11f0-8845-02420a0001ad'}, {'name': 'ARMSTRONG LOOKOUT 2', 'uuid': '574efdc2-1b23-11f0-8845-02420a0001ad'}, {'name': 'SIERRA AT TAHOE 1', 'uuid': '576baea4-1b23-11f0-8845-02420a0001ad'}, {'name': 'FALLEN LEAF LAKE 1', 'uuid': '5758b8b4-1b23-11f0-8845-02420a0001ad'}, {'name': 'HEAVENLY SKI AREA 1', 'uuid': '575aca61-1b23-11f0-8845-02420a0001ad'}, {'name': 'HEAVENLY SKI AREA 2', 'uuid': '575b214c-1b23-11f0-8845-02420a0001ad'}, {'name': 'RIDGE TAHOE NV 1', 'uuid': '57691d2e-1b23-11f0-8845-02420a0001ad'}, {'name': 'EDGEWOOD TAHOE RESORT 1', 'uuid': '57567220-1b23-11f0-8845-02420a0001ad'}, {'name': 'DL BLISS STATE PARK 1', 'uuid': '5757dea8-1b23-11f0-8845-02420a0001ad'}, {'name': 'BALD MTN NV 1', 'uuid': '5750

In [ ]:
# Pour sauvegarder le nouveau camera_table
# # --- Après avoir rempli camera_table ---

# output_file = "cameras_list.txt"

# with open(output_file, "w", encoding="utf-8") as f:
#     for cam in camera_table:
#         # On écrit chaque caméra sur une ligne : nom;uuid
#         f.write(f"{cam['name']};{cam['uuid']}\n")

# print(f"{len(camera_table)} caméras sauvegardées dans {output_file}")

123 caméras sauvegardées dans cameras_list.txt


In [ ]:
import requests
from PIL import Image
from io import BytesIO
import os

# Fichier contenant les noms et UUIDs
input_file = "cameras_list.txt"
base_folder = "./images_temp"

# Créer le dossier de base si nécessaire
os.makedirs(base_folder, exist_ok=True)

# Headers pour simuler un navigateur
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko)"
                  " Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://www.alertwildfire.org/"
}

# Lire le fichier
with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        name, uuid = line.split(";")
        if uuid == "None":
            continue  # Ignorer les caméras sans UUID

        # Créer un dossier pour la caméra
        cam_folder = os.path.join(base_folder, name.replace(" ", "_"))
        os.makedirs(cam_folder, exist_ok=True)

        # URL de l'image full
        img_url = f"https://s3-us-west-2.amazonaws.com/awf-data-public-prod/{uuid}/latest_full.jpg"

        try:
            response = requests.get(img_url, headers=headers, timeout=10)
            if response.status_code == 200:
                img = Image.open(BytesIO(response.content))
                save_path = os.path.join(cam_folder, f"{name.replace(' ', '_')}.jpg")
                img.save(save_path)
                print(f"Téléchargé : {save_path}")
            else:
                print(f"Erreur {response.status_code} pour {name}")
        except Exception as e:
            print(f"Erreur pour {name}: {e}")


In [22]:
camera_name = "Leek Spring"

# Trouver la caméra dans le JSON
camera_props = None
for feat in cameras_data.get("features", []):
    props = feat.get("properties", {})
    if camera_name in props.get("name") :
        camera_props = props
        break

if camera_props is None:
    raise ValueError(f"Caméra {camera_name} non trouvée dans le JSON")

print("Caméra trouvée dans le JSON :", camera_props)

Caméra trouvée dans le JSON : {'state': 'CA', 'az_current': '213.480000', 'fov': '62.98', 'fov_lft': ['-120.309995', ' 37.879450'], 'fov_rt': ['-121.143044', ' 38.308484'], 'fov_center': ['-120.802045', ' 38.002265'], 'region': 'AEU', 'lastupdate': 5, 'county': 'ElDorado', 'id': 'Axis-Leek', 'isp': 'NSL', 'attribution': 'USFS', 'name': 'Leek Springs', 'ptz': 1, 'ip': 'axis-leek', 'network': 'tahoe', 'tilt_current': '-11.55', 'ProdNbr': 'Q6055-E', 'zoom_current': '1.00'}


In [24]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import re

# Configurer Chrome headless
chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--disable-gpu")
driver = webdriver.Chrome(options=chrome_options)

# Construire l'URL du site pour la caméra
camera_url = f"https://www.alertwildfire.org/?currentFirecam={camera_props['ip']}&viewMode=Grid"
driver.get(camera_url)

# Récupérer l'élément <img> correspondant
img_elem = driver.find_element("css selector", "img.firecam-image")
img_src = img_elem.get_attribute("src")
print("src de l'image :", img_src)

# Extraire le camera_uuid depuis l'URL
match = re.search(r'/([a-f0-9\-]{36})/latest_full\.jpg', img_src)
if match:
    camera_uuid = match.group(1)
    print("Camera UUID trouvé :", camera_uuid)
else:
    print("Impossible de trouver le camera_uuid dans le src")

driver.quit()

NoSuchElementException: Message: no such element: Unable to locate element: {"method":"css selector","selector":"img.firecam-image"}
  (Session info: chrome=141.0.7390.123); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
	GetHandleVerifier [0x0x7ff7067fe8e5+80021]
	GetHandleVerifier [0x0x7ff7067fe940+80112]
	(No symbol) [0x0x7ff70658060f]
	(No symbol) [0x0x7ff7065d8854]
	(No symbol) [0x0x7ff7065d8b1c]
	(No symbol) [0x0x7ff70662c927]
	(No symbol) [0x0x7ff70660126f]
	(No symbol) [0x0x7ff70662968a]
	(No symbol) [0x0x7ff706601003]
	(No symbol) [0x0x7ff7065c95d1]
	(No symbol) [0x0x7ff7065ca3f3]
	GetHandleVerifier [0x0x7ff706abdc7d+2960429]
	GetHandleVerifier [0x0x7ff706ab7f3a+2936554]
	GetHandleVerifier [0x0x7ff706ad8977+3070247]
	GetHandleVerifier [0x0x7ff7068183ce+185214]
	GetHandleVerifier [0x0x7ff70681fe1f+216527]
	GetHandleVerifier [0x0x7ff706807b24+117460]
	GetHandleVerifier [0x0

In [18]:
import requests
from PIL import Image
from io import BytesIO

camera_uuid = "575ee718-1b23-11f0-8845-02420a0001ad"
img_url = f"https://s3-us-west-2.amazonaws.com/awf-data-public-prod/{camera_uuid}/latest_full.jpg"

# Headers simulant un navigateur
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko)"
                  " Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://www.alertwildfire.org/"
}

# Télécharger l'image
response = requests.get(img_url, headers=headers)
if response.status_code == 200:
    img = Image.open(BytesIO(response.content))
    img.show()  # ou img.save("leek_springs.jpg")
else:
    print("Erreur :", response.status_code)

In [3]:
# --- paramètres ---
n_samples = 5      # nombre d'acquisitions
interval = 30      # secondes entre acquisitions

# --- collecter les snapshots ---
snapshots = []
for i in range(n_samples):
    print(f"Fetching snapshot {i+1}/{n_samples}...")
    data = requests.get(CAMERAS_URL, headers=HEADERS).json()
    snapshots.append(data)
    if i < n_samples - 1:
        time.sleep(interval)

# --- comparer toutes les propriétés ---
changes = defaultdict(set)

for i in range(1, len(snapshots)):
    prev = snapshots[i-1]["features"]
    curr = snapshots[i]["features"]
    for f1, f2 in zip(prev, curr):
        props1, props2 = f1["properties"], f2["properties"]
        for key in props1:
            if props1[key] != props2.get(key):
                changes[key].add(f1["properties"]["id"])

# --- afficher résultats ---
print("\nChamps dynamiques détectés :")
for key, cams in changes.items():
    print(f" - {key} (modifié dans {len(cams)} caméras)")


Fetching snapshot 1/5...
Fetching snapshot 2/5...
Fetching snapshot 3/5...
Fetching snapshot 4/5...
Fetching snapshot 5/5...

Champs dynamiques détectés :


In [4]:
def process_camera_images(response, state, source, az_current):
    """
    Process and save camera images from an HTTP response into a specified path,
    based on the camera's state, source and azimut information.

    Args:
        response (requests.Response): The HTTP response containing the camera images.
        state (str): The state of the camera.
        source (str): The camera source identifier.
        az_current (str): The camera azimut.

    Logs:
        Error messages if any exception occurs during the processing.
    """

    local_time = get_camera_local_time(state)
    output_path = os.path.join(
        OUTPUT_BASE_PATH, "temp", local_time.strftime("%Y_%m_%d")
    )
    source_path = os.path.join(output_path, source)
    os.makedirs(source_path, exist_ok=True)

    try:
        for i, chunk in enumerate(generate_chunks(response)):
            output_path = os.path.join(source_path, f"{str(i).zfill(8)}.jpg")
            with open(output_path, "wb") as f:
                f.write(chunk)

        # Sort and rename images
        sort_and_rename_images(source_path, local_time)
    except Exception as e:
        logging.error(f"Error in processing images for {source_path}: {e}")

In [ ]:
import glob
import logging
import multiprocessing
import os
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta

import cv2
import numpy as np
import pytz
import requests
from dotenv import load_dotenv
from tqdm import tqdm


def download_and_process_images(cameras_features):
    """
    Download and process images for a list of cameras concurrently.

    Args:
        cameras_features (list): A list of camera features, each containing camera properties.
    """
    with ThreadPoolExecutor(max_workers=multiprocessing.cpu_count()) as executor, tqdm(
        total=len(cameras_features)
    ) as pbar:
        futures = []
        for cameras_feature in cameras_features:
            futures.append(
                executor.submit(
                    download_and_process_camera, cameras_feature["properties"]
                )
            )
        for future in as_completed(futures):
            result = future.result()
            pbar.update(1)
            if result:
                logging.error(result)

def download_and_process_camera(cam_properties):
    """
    Download and process camera images based on the camera properties. Handles errors such as timeouts.

    Args:
        cam_properties (dict): A dictionary containing properties of a camera, including its state and ID.
    """
    try:
        state = cam_properties.get("state")
        source = cam_properties.get("id").lower()
        region = cam_properties.get("region", "unknown")
        az_current = cam_properties.get("az_current", "0")


        url = f"https://ts1.alertwildfire.org/text/timelapse/?source={source}&preset={DURATION}"
        response = requests.get(url, headers=HEADERS, timeout=MAX_TIME)
        process_camera_images(response, state, source, region, az_current)
    except requests.exceptions.Timeout:
        logging.error(f"Timeout processing {source}")
    except Exception as e:
        logging.error(f"Error processing {source}: {e}")
        
def process_camera_images(response, state, source, region, az_current):
    """
    Process and save camera images, organized by region, source, and azimuth.
    """
    local_time = get_camera_local_time(state)
    date_folder = local_time.strftime("%Y_%m_%d")
    source_path = os.path.join(
        OUTPUT_BASE_PATH, region, source, f"az_{az_current}", date_folder
    )
    os.makedirs(source_path, exist_ok=True)

    try:
        for i, chunk in enumerate(generate_chunks(response)):
            img_path = os.path.join(source_path, f"{str(i).zfill(8)}.jpg")
            with open(img_path, "wb") as f:
                f.write(chunk)

        sort_and_rename_images(source_path, local_time)
    except Exception as e:
        logging.error(f"Error processing {source}: {e}")

def sort_and_rename_images(source_path, local_time):
    """
    Sort and rename images in a directory based on their timestamps relative to the local time.

    Args:
        source_path (str): The directory containing the images.
        local_time (datetime): The local time of the camera when the first image was captured.

    Logs:
        Error messages if any exception occurs during the sorting and renaming process.
    """

    try:
        imgs = glob.glob(os.path.join(source_path, "*"))
        imgs.sort()
        nb_imgs = len(imgs)

        if nb_imgs > 0:
            dt = (
                duration_to_seconds(DURATION) / nb_imgs
            )  # Total duration divided by the number of images
            local_time = local_time - timedelta(
                hours=duration_to_seconds(DURATION) / 3600
            )

            for i, file in enumerate(imgs):
                frame_time = local_time + timedelta(seconds=dt * i)
                frame_name = frame_time.strftime("%Y_%m_%dT%H_%M_%S") + ".jpg"
                new_file = os.path.join(source_path, frame_name)
                shutil.move(file, new_file)
    except Exception as e:
        logging.error(f"Error in sorting and renaming images in {source_path}: {e}")

def get_camera_local_time(state):
    """
    Get the local time for a given state using the specified state's timezone.

    Args:
        state (str): The state for which to find the local time.

    Returns:
        datetime: The current local time for the given state.
    """
    timezone_str = STATE_TIMEZONES.get(
        state, "America/Phoenix"
    )  # Default to America/Phoenix if state not found
    timezone = pytz.timezone(timezone_str)
    return datetime.now(timezone)

STATE_TIMEZONES = {
    "AZ": "America/Phoenix",  # Arizona
    "CA": "America/Los_Angeles",  # California
    "CO": "America/Denver",  # Colorado
    "ID": "America/Boise",  # Idaho
    "MT": "America/Denver",  # Montana
    "NV": "America/Los_Angeles",  # Nevada
    "Nevada": "America/Los_Angeles",  # Alternate name for Nevada
    "OR": "America/Los_Angeles",  # Oregon
    "WA": "America/Los_Angeles",  # Washington
    # Add other states and their timezones here
}

def duration_to_seconds(duration_str):
    """
    Convert a duration string to the equivalent number of seconds.

    Args:
        duration_str (str): A duration string in the format "Xh" (X hours) or "Xmn" (X minutes).

    Returns:
        int: The number of seconds equivalent to the duration string.

    Raises:
        ValueError: If the duration string format is invalid.
    """
    duration_str = duration_str.lower()
    if duration_str.endswith("h"):
        hours = int(duration_str[:-1])
        return hours * 60 * 60
    elif duration_str.endswith("mn"):
        minutes = int(duration_str[:-2])
        return minutes * 60
    else:
        raise ValueError("Invalid duration string format")
    
def generate_chunks(response):
    """
    Splits the response content into chunks based on a specific delimiter. Each chunk represents a frame or segment.

    Args:
        response (requests.Response): The HTTP response containing the content to be split.

    Yields:
        bytes: A chunk of the response content.
    """

    chunks = response.content.split(b"--frame\r\n")
    for chunk in chunks:
        if len(chunk) > 100:
            start = chunk.find(b"\xff\xd8")
            yield chunk[start:]


def get_camera_local_time(state):
    """
    Get the local time for a given state using the specified state's timezone.

    Args:
        state (str): The state for which to find the local time.

    Returns:
        datetime: The current local time for the given state.
    """
    timezone_str = STATE_TIMEZONES.get(
        state, "America/Phoenix"
    )  # Default to America/Phoenix if state not found
    timezone = pytz.timezone(timezone_str)
    return datetime.now(timezone)

In [9]:
download_and_process_images(cameras_data["features"])

100%|██████████| 1055/1055 [03:24<00:00,  5.16it/s]
